# BDAT05 · MIDTERM · Part 2A — Practical Notebook C
**Panaderia Buena Masa** — a Laguna bakeshop chain you have never seen before.
**60 minutes · open notes (your own notebooks and code) · NO AI of any kind.**

Five sections. Every section ends with a printed **EVIDENCE** result and a one-sentence **READING**.
Nothing changes silently — print your counts. A correct number with no reading earns nothing.

In [5]:
# SETUP — run first. Your 60 minutes start here.
from datetime import datetime
from zoneinfo import ZoneInfo
STUDENT_NAME = "Lagdamen, Lawrence M"      # Surname, First Name
NOTEBOOK = "C"
START_TIME = datetime.now(ZoneInfo("Asia/Manila"))
assert STUDENT_NAME.strip(), "Type your name before anything else."
print("Examinee:", STUDENT_NAME, "| Notebook", NOTEBOOK, "| started", START_TIME.strftime("%Y-%m-%d %H:%M"))

Examinee: Lagdamen, Lawrence M | Notebook C | started 2026-10-03 15:05


In [13]:
# LOAD — paste the four FILE_IDs from the access sheet, then run.
import pandas as pd, numpy as np
FILE_ID_ORDERS   = "1DIb2i76B0vXE3NfdtNGTfnVyUtRUzfc3"
FILE_ID_BRANCHES = "1Wm5C-_c3X2FJy5P7wzTn2qNgLsXQsWii"
FILE_ID_CALENDAR = "1YXJjRoLypaIWRfuZArsKpmbBv7bObxqw"
FILE_ID_PREDS    = "1mqUkfUOaquX8ERARpH7ipoIHgfPNKVi0"
def _url(fid): return f"https://drive.google.com/uc?export=download&id={fid}"
orders   = pd.read_csv(_url(FILE_ID_ORDERS), dtype={"AmountPHP": str}, keep_default_na=False)
branches = pd.read_csv(_url(FILE_ID_BRANCHES))
calendar = pd.read_csv(_url(FILE_ID_CALENDAR), parse_dates=["Date"])
preds    = pd.read_csv(_url(FILE_ID_PREDS), parse_dates=["Date"])
print(orders.shape, branches.shape, calendar.shape, preds.shape)
orders.head(3)

(42028, 7) (6, 5) (608, 5) (552, 3)


,OrderID,Date,BranchID,ItemsSold,Channel,Payment,AmountPHP
0,BM108937,2025-12-16,P03,4,Store,Cash,67.33
1,BM101926,2025-05-22,P05,3,GrabFood,Cash,176.22
2,BM120165,2026-05-27,P05,4,Store,GCash,80.41


---
## §1 · Prepare, then split — the calendar does the splitting  *(10 points)*

The file carries **five known defect families**: duplicated orders, dirty BranchID variants,
text-trapped amounts, blank dates, and orphan branch rows. Four are repaired for you below —
run and check the printed counts. **Your spotlight defect: text-trapped amounts.**
AmountPHP loads as text, not numbers. Find what is trapping it, count the culprit values, and recover the column to numeric without dropping a single row.

Then split by the calendar: **training = everything before 2026-06-01 · holdout = 2026-06-01 onward.**

**EVIDENCE:** the count for your spotlight repair, the clean row count, and the train / holdout branch-day counts.
**READING:** one sentence — why does the calendar make this split, and not `sample()`?

In [27]:
# GIVEN repairs — four of the five families. Run and check each count.
work = orders.copy()
print("raw rows:", len(work))

work = work.drop_duplicates(subset=["OrderID"])
print("after dedupe:", len(work))

V = {"p01":"P01","P-02":"P02","P03.":"P03","p04":"P04","P 05":"P05","P06 ":"P06"}
work["BranchID"] = work["BranchID"].replace(V)
print("branch codes now:", sorted(work["BranchID"].unique()))

work = work[work["Date"] != ""]
work["Date"] = pd.to_datetime(work["Date"])
print("after date repair:", len(work))

work = work[work["BranchID"].isin(branches["BranchID"])]
print("after orphan removal:", len(work))

raw rows: 42028
after dedupe: 42000
branch codes now: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07']
after date repair: 41984
after orphan removal: 41963


In [28]:
# YOUR spotlight repair — text-trapped amounts.
# Investigate, count, repair. Print the count BEFORE you repair it.
# ...your code...

# inspect the trapped values first
amount_num = pd.to_numeric(work["AmountPHP"], errors="coerce")

bad_amounts = work.loc[
    amount_num.isna() & work["AmountPHP"].notna(),
    "AmountPHP"
]

print("trapped rows:", len(bad_amounts))
print(bad_amounts.value_counts())

# repair common text traps, then convert to numeric
work["AmountPHP"] = (
    work["AmountPHP"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₱", "", regex=False)
    .str.strip()
)

work["AmountPHP"] = pd.to_numeric(work["AmountPHP"], errors="coerce")

print("AmountPHP dtype:", work["AmountPHP"].dtype)
print("rows after amount repair:", len(work))
print("missing AmountPHP after repair:", work["AmountPHP"].isna().sum())

trapped rows: 18
AmountPHP
10,385.12    1
11,238.76    1
16,215.57    1
11,397.14    1
17,657.58    1
12,133.65    1
18,123.89    1
17,155.59    1
12,906.72    1
15,152.60    1
16,336.87    1
15,091.93    1
12,941.71    1
7,256.47     1
10,666.78    1
7,982.87     1
10,550.25    1
13,094.16    1
Name: count, dtype: int64
AmountPHP dtype: float64
rows after amount repair: 41963
missing AmountPHP after repair: 0


In [29]:
# EVIDENCE §1 — run after your repair. All five families must be repaired by this point.
clean = work.copy()
print("EVIDENCE clean rows:", len(clean))
bd = clean.groupby(["BranchID","Date"], as_index=False)["AmountPHP"].sum().rename(columns={"AmountPHP":"Revenue"})
SPLIT = pd.Timestamp("2026-06-01")
train, hold = bd[bd["Date"] < SPLIT], bd[bd["Date"] >= SPLIT]
print("EVIDENCE branch-days:", len(bd), "| train:", len(train), "| holdout:", len(hold))

EVIDENCE clean rows: 41963
EVIDENCE branch-days: 3647 | train: 3095 | holdout: 552


**READING §1:**  
*(one sentence — why the calendar, not `sample()`)*

> The calendar split keeps future data out of training and matches real forecasting, while sample() could mix future observations into the training set and cause leakage.

---
## §2 · Features and the leakage test  *(15 points)*

The test is one question: **was this knowable at the moment the decision would be taken?**
The decision: *predict a branch-day's revenue, the evening before.*

**(a) Build your assigned feature — `PromoPlanned`.** Join the calendar and build PromoPlanned for each branch-day. EVIDENCE: mean revenue on promo-planned days vs ordinary days.

**(b) The candidate table.** Classify each candidate **KEEP** or **LEAK**, with a calendar reason.

| # | Candidate feature | What the column is |
|---|---|---|
| 1 | DayOfWeek of the day being predicted | calendar fact, fixed in advance |
| 2 | PromoPlanned flag for the day being predicted | planned by head office weeks ahead |
| 3 | The same branch's revenue 7 days earlier | already finished and recorded |
| 4 | ItemsSold on the day being predicted | counted only as the day happens |
| 5 | Month-to-date revenue including the day being predicted | contains the target day inside the sum |
| 6 | Refunds processed on the day being predicted | processed during and after the day |


In [30]:
# (a) Build PromoPlanned and print its EVIDENCE.
# ...your code...
print(calendar.columns.tolist())
calendar.head()

['Date', 'DayOfWeek', 'IsWeekend', 'IsHoliday', 'PromoPlanned']


,Date,DayOfWeek,IsWeekend,IsHoliday,PromoPlanned
0,2025-01-01,Wednesday,False,True,False
1,2025-01-02,Thursday,False,False,False
2,2025-01-03,Friday,False,False,False
3,2025-01-04,Saturday,True,False,False
4,2025-01-05,Sunday,True,False,False


**(b) Verdicts — fill every row:**

| # | KEEP or LEAK | Calendar reason (one clause) |
|---|---|---|
| 1 | KEEP | Day of week is known before the day starts. |
| 2 | KEEP | The promotion is planned weeks in advance. |
| 3 | KEEP | Revenue from 7 days earlier has already been recorded. |
| 4 | LEAK | Items sold are only known as the target day happens. |
| 5 | LEAK | The total already includes revenue from the day being predicted. |
| 6 | LEAK | Refunds are processed during or after the target day. |

**READING §2:** *(one sentence — the feature test, in your own words)*

> A feature is safe only if I could already know it the evening before making the prediction.

---
## §3 · Baseline first — you are not allowed to beat nothing  *(10 points)*

Before looking at any model: compute the **naive-7 baseline** — predict each holdout branch-day
with the same branch's revenue **7 days earlier** — and score it with **MAE** on both of your assigned branches.
Your assigned slice: **P05 and P03**.

**EVIDENCE:** the baseline MAE, printed per assigned slice.
**READING:** one sentence — why is this baseline computed *before* the model is examined?

In [31]:
# §3 — naive-7 baseline on the holdout.
ASSIGNED = ['P05', 'P03']
bdi = bd.set_index(["BranchID","Date"])["Revenue"]
ho = hold.copy()
ho["Naive7"] = [bdi.get((r.BranchID, r.Date - pd.Timedelta(days=7)), np.nan) for r in ho.itertuples()]
ho = ho.dropna(subset=["Naive7"])
def mae(a, f): return float(np.mean(np.abs(a - f)))
for tag in ASSIGNED:
    g = ho if tag == "ALL" else ho[ho["BranchID"] == tag]
    print(f"EVIDENCE baseline MAE [{tag}]:", round(mae(g["Revenue"], g["Naive7"]), 2))

EVIDENCE baseline MAE [P05]: 679.26
EVIDENCE baseline MAE [P03]: 738.82


**READING §3:**

> The baseline is computed before examining the model so we have a fair benchmark to beat and cannot choose an easier comparison afterward.

---
## §4 · The model must earn it  *(15 points)*

Head office bought **“Model M,”** a forecasting service. Its predictions for every holdout
branch-day are in `preds`. Score Model M with MAE on your assigned slice —
**P05 and P03** — beside your §3 baseline, and compute the ratio
**MAE(Model M) ÷ MAE(naive-7)**. Below 1.0 = the model earns its keep on that slice.

**EVIDENCE:** both MAEs and the ratio, per assigned slice, with **WINS / LOSES** printed.
**READING:** one sentence per slice — what the ratio says, in pesos-and-decisions language.

In [32]:
# §4 — Model M vs your baseline.
ev = ho.merge(preds, on=["BranchID","Date"], how="left").dropna(subset=["ModelM_PredictedRevenue"])
for tag in ASSIGNED:
    g = ev if tag == "ALL" else ev[ev["BranchID"] == tag]
    m_b = mae(g["Revenue"], g["Naive7"]); m_m = mae(g["Revenue"], g["ModelM_PredictedRevenue"])
    r = m_m / m_b
    print(f"EVIDENCE [{tag}] baseline {m_b:,.2f} | Model M {m_m:,.2f} | ratio {r:.3f} |", "MODEL WINS" if r < 1 else "MODEL LOSES")

EVIDENCE [P05] baseline 679.26 | Model M 539.32 | ratio 0.794 | MODEL WINS
EVIDENCE [P03] baseline 738.82 | Model M 849.89 | ratio 1.150 | MODEL LOSES


**READING §4:**

> P05: Model M wins because its average error drops from ₱679.26 to ₱539.32, so it gives head office a more accurate revenue estimate for staffing and stock decisions.
P03: Model M loses because its average error rises from ₱738.82 to ₱849.89, so the simple naive-7 forecast is actually safer for decisions on this branch.

---
## §5 · Judgment — the memo line  *(5 points)*

Head office asks one question: **“Do we roll Model M out to every branch?”**

Write your verdict in **three sentences maximum**, using only numbers your own notebook printed:
what you recommend, which evidence carries it, and what you would check next.
A verdict your own evidence does not support earns nothing.

**MEMO:**

> Do not roll Model M out to every branch yet. It improved P05 from ₱679.26 to ₱539.32 (ratio 0.794), but worsened P03 from ₱738.82 to ₱849.89 (ratio 1.150), so performance is not consistent across branches. Next, I would test Model M against the naive-7 baseline on the other branches before deciding where it should be deployed.

In [33]:
# FINISH — run last.
from datetime import datetime
from zoneinfo import ZoneInfo
END_TIME = datetime.now(ZoneInfo("Asia/Manila"))
def _surname(name):
    if "," in name: return name.split(",")[0].strip().upper()
    parts = [p for p in name.split() if p.strip(".")]
    return (parts[-1] if parts else "SURNAME").upper()
mins = (END_TIME - START_TIME).total_seconds() / 60
print(f"Examinee: {STUDENT_NAME} | Notebook {NOTEBOOK} | elapsed {mins:.0f} min")
print(f"Save as: BDAT05_Midterm_Part2A_{NOTEBOOK}_{_surname(STUDENT_NAME)}.ipynb and upload to your midterm folder.")

Examinee: Lagdamen, Lawrence M | Notebook C | elapsed 43 min
Save as: BDAT05_Midterm_Part2A_C_LAGDAMEN.ipynb and upload to your midterm folder.
